In [ ]:
# interpretability_analysis.ipynb

# Cell 1: Setup
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from configs.config import Config
from models.bu_net_multitask import BUNetMultiTask

# Load your best baseline model
baseline_model = BUNetMultiTask(
    encoder_name='resnet34',
    in_channels=2,
    num_classes=4,
    num_tumor_types=5
).to(Config.DEVICE)

checkpoint = torch.load('./checkpoints/best_model.pth', weights_only=False)
baseline_model.load_state_dict(checkpoint['model_state_dict'])
baseline_model.eval()

print("Model loaded for interpretability analysis")

In [ ]:
# Cell 2: Extract and visualize encoder features
def visualize_feature_maps(model, image, layer_name='encoder'):
    """Visualize what features the encoder extracts"""
    
    # Hook to capture activations
    activations = {}
    def hook_fn(name):
        def hook(module, input, output):
            activations[name] = output.detach()
        return hook
    
    # Register hooks on encoder layers
    hooks = []
    for i, layer in enumerate(model.unet.encoder.children()):
        hook = layer.register_forward_hook(hook_fn(f'layer_{i}'))
        hooks.append(hook)
    
    # Forward pass
    with torch.no_grad():
        _ = model(image.unsqueeze(0).to(Config.DEVICE))
    
    # Remove hooks
    for hook in hooks:
        hook.remove()
    
    return activations

# Test on one sample
from data.dataset import BraTSDataset
sample = val_dataset[50]  # Pick one with tumor
image = sample['image']
mask = sample['mask']

activations = visualize_feature_maps(baseline_model, image)

# Visualize first few channels of each layer
fig, axes = plt.subplots(len(activations), 8, figsize=(16, 2*len(activations)))

for layer_idx, (name, act) in enumerate(activations.items()):
    act_cpu = act.cpu().squeeze(0)  # Remove batch dim
    
    for channel in range(min(8, act_cpu.shape[0])):
        ax = axes[layer_idx, channel] if len(activations) > 1 else axes[channel]
        ax.imshow(act_cpu[channel], cmap='viridis')
        ax.axis('off')
        if channel == 0:
            ax.set_title(f'{name}', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(Config.RESULTS_DIR, 'feature_maps.png'), dpi=150)
plt.show()

In [ ]:
# Cell 3: Visualize attention weights 
def visualize_attention(model, image, mask):
    """Visualize where the model pays attention"""
    
    model.eval()
    
    with torch.no_grad():
        # Forward with return_features=True
        seg_out, cls_out, features, attention = model(
            image.unsqueeze(0).to(Config.DEVICE), 
            return_features=True
        )
    
    # Plot
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    # Original FLAIR
    axes[0].imshow(image[0].cpu(), cmap='gray')
    axes[0].set_title('Input (FLAIR)')
    axes[0].axis('off')
    
    # Ground truth
    axes[1].imshow(mask.cpu(), cmap='viridis')
    axes[1].set_title('Ground Truth')
    axes[1].axis('off')
    
    # Prediction
    pred = torch.argmax(seg_out, dim=1).squeeze().cpu()
    axes[2].imshow(pred, cmap='viridis')
    axes[2].set_title('Prediction')
    axes[2].axis('off')
    
    # Attention weights (averaged across channels)
    attention_map = attention.squeeze().mean(dim=0).cpu()
    im = axes[3].imshow(attention_map, cmap='hot')
    axes[3].set_title('Attention Map')
    axes[3].axis('off')
    plt.colorbar(im, ax=axes[3])
    
    plt.tight_layout()
    plt.show()

# Visualize for a few samples
for i in [10, 50, 100]:
    sample = val_dataset[i]
    visualize_attention(baseline_model, sample['image'], sample['mask'])